In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

%cd /content/drive/MyDrive/LLM/


In [ ]:
%cd /content/drive/MyDrive/

In [3]:
# If running on Colab, install minimal deps
# (Comment out torch install if Colab runtime already has a good CUDA build.)
# %pip install -q --upgrade transformers accelerate

import os, sys, platform, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.8.0+cu126
CUDA available: True


In [4]:
!pip install llamafactory

In [5]:
#!pip -q install --upgrade "transformers>=4.43" accelerate safetensors sentencepiece einops
import torch, platform, sys, os, textwrap
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


Torch: 2.8.0+cu126 | CUDA: True | Device: NVIDIA A100-SXM4-80GB


In [6]:
# Versions + sanity checks (run after installing deps)

import sys, os, platform, tempfile, warnings
import torch
from packaging import version

# Core libs
import transformers, accelerate, safetensors, sentencepiece, einops

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("Torch:", torch.__version__, "| CUDA avail:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU CC:", f"{props.major}.{props.minor}", "| Total VRAM (GB):", round(props.total_memory/1024**3, 2))

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("safetensors:", safetensors.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("einops:", einops.__version__)

# Assert min versions you requested
assert version.parse(transformers.__version__) >= version.parse("4.43"), "Transformers >= 4.43 required"


Python: 3.12.12 | Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Torch: 2.8.0+cu126 | CUDA avail: True | CUDA: 12.6
GPU: NVIDIA A100-SXM4-80GB
GPU CC: 8.0 | Total VRAM (GB): 79.32
Transformers: 4.52.4
Accelerate: 1.7.0
safetensors: 0.6.2
sentencepiece: 0.2.1
einops: 0.8.1


In [ ]:
BASE_MODEL_NAME        = "Qwen/Qwen2.5-0.5B-Instruct"     # LLM backbone
PROTEIN_CONFIG = "protrek/weights/ProTrek_35M/esm2_t12_35M_UR50D"
STRUCTURE_CONFIG = "protrek/weights/ProTrek_35M/foldseek_t12_35M"
PROTREK_CKPT_PATH    = "weights/ProTrek_35M/ProTrek_35M.pt"


DTYPE = "bf16" if torch.cuda.is_available() else "fp16"
print("Using dtype:", DTYPE)


Using dtype: bf16


In [8]:
#import protein_llm  # this file must define class `PLLM`
import protein_llm.models.modeling_pllm

[WARN] flash_attn is not installed, flash_attn will not work


In [9]:
import os, json, math
from typing import Optional, List
import torch
from transformers import AutoTokenizer

# Import your updated PLLM module (must be in the same folder)
import protein_llm  # this file must define class `PLLM`
import protein_llm.models.modeling_pllm as PLLM
from protein_llm.models.configuration_pllm import PLLMConfig

# ===== USER-EDITABLE PATHS =====
#BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # or "Qwen/Qwen2.5-0.5B"
PROT_SLOT = 1
STRU_SLOT = 3

# BF16 is optimal for A100
DTYPE_STR = "bf16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
tokenizer.add_tokens("<protein>")
tokenizer.add_tokens("<structure>")
protein_token_id = tokenizer("<protein>", add_special_tokens=False).input_ids[-1]
structure_token_id = tokenizer("<structure>", add_special_tokens=False).input_ids[-1]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
cfg = PLLMConfig(
    base_model_name_or_path=BASE_MODEL_NAME,
    protein_config=PROTEIN_CONFIG,
    structure_config=STRUCTURE_CONFIG,
    protrek_ckpt=PROTREK_CKPT_PATH if PROTREK_CKPT_PATH else None,
    prot_slot=1, stru_slot=3,
    single_token_prefix=False, prefix_len=4,
    proj_hid=1024, dropout=0.10,
    train_encoders=False,        # freeze encoders for the test
    load_pretrained=True,        # load base LLM from hub
    protein_token_id=protein_token_id,
    structure_token_id=structure_token_id,
)


In [12]:
pllm = PLLM.PLLM(cfg)
pllm.load_protrek_weights()
pllm = pllm.to("cuda")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[PLLM] Loaded generation_config from base model: Qwen/Qwen2.5-0.5B-Instruct
[ProteinEncoder] loaded from slot 1 | missing=[] unexpected=[]
[StructureEncoder] loaded from slot 3 | missing=[] unexpected=[]


In [13]:
user_prompt = (
    "You are a professional protein biologist. "
    "Based only on the provided inputs, produce a natural, concise, and biologically accurate description of the protein. "
    "First reason step by step inside a <thinking> block using sequence-derived evidence and structural cues, "
    "then provide the final 2–4 sentence description inside an <answer> block.\n\n"
    "Inputs:\n"
    "Protein: <protein>\n"
    "Structure: <structure>"
)

aa_seq = "MSKGTPSRGKRQTQTHLTCRRCGRMSYHKRHKICSSCGFGRSTRMRSYGWITKRPKVATH"
structure = "<|chain:A|> <|chain_sep|> #ddddvvvvpppddqfdqdppprdraqgpvqragpqqggpndpggdddpvvddddpdddd"


In [14]:
enc = tokenizer(user_prompt, return_tensors="pt")
input_ids = enc["input_ids"].to(DEVICE)
attention_mask = enc["attention_mask"].to(DEVICE)

print("Prompt length:", input_ids.shape[-1])
print("Has <protein>:", (input_ids == cfg.protein_token_id).any().item())
print("Has <structure>:", (input_ids == cfg.structure_token_id).any().item())

Prompt length: 77
Has <protein>: True
Has <structure>: True


In [15]:
chat_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": user_prompt}],
    add_generation_prompt=True,
    tokenize=False,
)

# Make sure the placeholders are present after templating
assert "<protein>" in chat_prompt and "<structure>" in chat_prompt, "Placeholders not found in chat prompt."

test_inputs = tokenizer(chat_prompt, return_tensors="pt").to(DEVICE)

# --- Generate (condition on aa_seq & structure) ---
pllm.eval()
with torch.no_grad():
    gen_ids = pllm.generate(
        **test_inputs,
        aa_seq=[aa_seq],
        stru_str=[structure],
        max_new_tokens=512,
        do_sample=False,      # deterministic for debugging; change if you want sampling
        temperature=1.0,
        top_p=1.0,
    )

# --- Strip the prompt tokens from the output and decode ---
trimmed = [
    output_ids[len(inp_ids):]
    for inp_ids, output_ids in zip(test_inputs.input_ids, gen_ids)
]
response = tokenizer.batch_decode(trimmed, skip_special_tokens=True)[0]

print("=== MODEL RESPONSE ===")
print(response)
print("\n(first 64 generated token ids):")
print(trimmed[0][:64].tolist())

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== MODEL RESPONSE ===
<thinking>
The provided sequence is a short protein sequence with a length of 16 amino acids. This sequence does not appear to be part of any known protein family or structure. It could potentially belong to a non-coding RNA or regulatory region.
</thinking>

<answer>
A short protein sequence consisting of 16 amino acids appears to lack a clear biological function or context within the given information. It could potentially represent a non-coding RNA or regulatory region, but without more context, it's difficult to determine its role in biology. Further analysis would be needed to identify its specific role or significance.

(first 64 generated token ids):
[27, 82260, 397, 785, 3897, 8500, 374, 264, 2805, 12833, 8500, 448, 264, 3084, 315, 220, 16, 21, 41400, 32869, 13, 1096, 8500, 1558, 537, 4994, 311, 387, 949, 315, 894, 3881, 12833, 2997, 476, 5944, 13, 1084, 1410, 13581, 9173, 311, 264, 2477, 1786, 3700, 40114, 476, 22515, 5537, 624, 522, 82260, 1339, 27, 921

In [16]:
pllm.eval()
with torch.no_grad():
    out = pllm(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=None,
        aa_seq=[aa_seq],
        stru_str=[structure],
        extract_qkv=True,
        layer_idx=[0],
    )

print("Forward OK. Cached tensors ready for export.")


Forward OK. Cached tensors ready for export.


In [17]:

exp = pllm.export_qkv(split_heads=False, to_cpu=False)

assert "layers" in exp and "m" in exp, "export must contain 'layers' and 'm'"
assert 0 in exp["layers"], "layer 0 must be present in exported layers"

q = exp["layers"][0]["q"]
k = exp["layers"][0]["k"]
v = exp["layers"][0]["v"]
m = exp["m"]

print("Single-layer (fused) shapes:")
print("  q:", tuple(q.shape), "k:", tuple(k.shape), "v:", tuple(v.shape))
print("  mask:", None if m is None else tuple(m.shape))
print("  meta:", exp["layers"][0]["meta"])

B, S, Hq = q.shape
_, Sk, Hk = k.shape
_, Sv, Hv = v.shape
assert S == Sk == Sv, "sequence lengths mismatch"
assert q.requires_grad is False and k.requires_grad is False and v.requires_grad is False, "exported QKV should be detached"

Single-layer (fused) shapes:
  q: (1, 213, 896) k: (1, 213, 128) v: (1, 213, 128)
  mask: (1, 213)
  meta: {'layer_idx': 0, 'split_heads': False, 'num_heads': 14, 'num_key_value_heads': 2, 'head_dim_q': None, 'head_dim_kv': None, 'seq_len': 213}


In [18]:
exp_split = pllm.export_qkv(split_heads=True, to_cpu=True)  # move to CPU to free VRAM
layer0 = exp_split["layers"][0]
qh, kh, vh = layer0["q"], layer0["k"], layer0["v"]
meta = layer0["meta"]

print("Single-layer (split heads) shapes:")
print("  q:", tuple(qh.shape), "k:", tuple(kh.shape), "v:", tuple(vh.shape))
print("  meta:", meta)

# GQA
Nh   = meta["num_heads"]
Nkv  = meta["num_key_value_heads"] if meta["num_key_value_heads"] is not None else Nh
hd_q = meta["head_dim_q"]
hd_kv= meta["head_dim_kv"]

assert qh.shape[2] == Nh and qh.shape[3] == hd_q, "Q head split mismatch"
assert kh.shape[2] == Nkv and kh.shape[3] == hd_kv, "K head split mismatch"
assert vh.shape[2] == Nkv and vh.shape[3] == hd_kv, "V head split mismatch"

assert Nh % Nkv == 0, "num_heads must be a multiple of num_key_value_heads for GQA broadcasting"
print("Broadcast factor Nh/Nkv =", Nh // Nkv)


Single-layer (split heads) shapes:
  q: (1, 213, 14, 64) k: (1, 213, 2, 64) v: (1, 213, 2, 64)
  meta: {'layer_idx': 0, 'split_heads': True, 'num_heads': 14, 'num_key_value_heads': 2, 'head_dim_q': 64, 'head_dim_kv': 64, 'seq_len': 213}
Broadcast factor Nh/Nkv = 7


In [19]:

layers_to_probe = [0, 1, 2]
pllm.eval()
with torch.no_grad():
    out = pllm(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=None,
        aa_seq=[aa_seq],
        stru_str=[structure],
        extract_qkv=True,
        layer_idx=layers_to_probe,
    )

print(f"Forward OK. QKV captured for layers {layers_to_probe}.")


Forward OK. QKV captured for layers [0, 1, 2].


In [20]:
exp_multi = pllm.export_qkv(split_heads=False, to_cpu=False)
layers = exp_multi["layers"]
m = exp_multi["m"]

print("Exported layers:", list(layers.keys()))
print("mask shape:", None if m is None else tuple(m.shape))

for li, blob in layers.items():
    q, k, v = blob["q"], blob["k"], blob["v"]
    print(f"Layer {li}: q {tuple(q.shape)} | k {tuple(k.shape)} | v {tuple(v.shape)}")
    # sanity: same seq lens
    assert q.shape[1] == k.shape[1] == v.shape[1], f"seq length mismatch at layer {li}"

Exported layers: [0, 1, 2]
mask shape: (1, 213)
Layer 0: q (1, 213, 896) | k (1, 213, 128) | v (1, 213, 128)
Layer 1: q (1, 213, 896) | k (1, 213, 128) | v (1, 213, 128)
Layer 2: q (1, 213, 896) | k (1, 213, 128) | v (1, 213, 128)


In [21]:
exp_multi_split = pllm.export_qkv(split_heads=True, to_cpu=True)
layers = exp_multi_split["layers"]

for li, blob in layers.items():
    qh, kh, vh = blob["q"], blob["k"], blob["v"]
    meta = blob["meta"]
    Nh   = meta["num_heads"]
    Nkv  = meta["num_key_value_heads"] if meta["num_key_value_heads"] is not None else Nh
    hd_q = meta["head_dim_q"]
    hd_kv= meta["head_dim_kv"]

    print(f"[Layer {li}] q {tuple(qh.shape)} | k {tuple(kh.shape)} | v {tuple(vh.shape)} | Nh={Nh} Nkv={Nkv}")
    assert qh.shape[2] == Nh and qh.shape[3] == hd_q, "Q split mismatch"
    assert kh.shape[2] == Nkv and kh.shape[3] == hd_kv, "K split mismatch"
    assert vh.shape[2] == Nkv and vh.shape[3] == hd_kv, "V split mismatch"
    assert Nh % Nkv == 0, "GQA broadcasting factor must be integer"


[Layer 0] q (1, 213, 14, 64) | k (1, 213, 2, 64) | v (1, 213, 2, 64) | Nh=14 Nkv=2
[Layer 1] q (1, 213, 14, 64) | k (1, 213, 2, 64) | v (1, 213, 2, 64) | Nh=14 Nkv=2
[Layer 2] q (1, 213, 14, 64) | k (1, 213, 2, 64) | v (1, 213, 2, 64) | Nh=14 Nkv=2


In [23]:
m = exp_multi["m"]
any_mask = m is not None
print("Mask exists:", any_mask)
if any_mask:
    one_layer = next(iter(exp_multi["layers"].values()))
    S = one_layer["q"].shape[1]
    assert m.shape[1] == S, f"Mask length {m.shape[1]} should match QKV seq len {S}"
    print("Mask length matches QKV seq length:", S)


Mask exists: True
Mask length matches QKV seq length: 213
